# COVID-19 Indonesia Data Analysis
**Dataset:** COVID-19 Indonesia (Kaggle)  
**Author:** Fikri Firstly Arrasyid Hawe  
**Goal:** Time-series and provincial analysis of COVID-19 spread, mortality, and recovery across Indonesia.

---
### Setup
Run `pip install kagglehub pandas matplotlib seaborn` before starting.

In [ ]:
import kagglehub
import os, pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

path = kagglehub.dataset_download('hendratno/covid19-indonesia')
print('Files:', os.listdir(path))

csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
df = pd.read_csv(os.path.join(path, csv_files[0]))
print(f'Shape: {df.shape}')
df.head()

## 1. Data Overview

In [ ]:
print(df.columns.tolist())
print(df.dtypes)

# Identify date column
date_col = [c for c in df.columns if 'date' in c.lower() or 'tanggal' in c.lower()][0]
df[date_col] = pd.to_datetime(df[date_col])
print(f'Date range: {df[date_col].min()} to {df[date_col].max()}')

## 2. National Time-Series Trends

In [ ]:
# Try to find national-level data
case_cols = [c for c in df.columns if any(k in c.lower() for k in ['total_case', 'total_death', 'total_recov', 'kasus', 'meninggal', 'sembuh'])]
print('Relevant columns:', case_cols)

# Use first 3 relevant numeric cols as proxy if exact names differ
numeric_cols = df.select_dtypes(include='number').columns.tolist()
print('Numeric columns:', numeric_cols[:10])

In [ ]:
# Aggregate nationally by date
national = df.groupby(date_col)[numeric_cols[:6]].sum().reset_index()

# Plot the first 3 numeric metrics as trend lines
fig, axes = plt.subplots(3, 1, figsize=(14, 12))
colors = ['#1a1a1a', '#c8a87a', '#e8d5b7']

for i, (col, ax, color) in enumerate(zip(numeric_cols[:3], axes, colors)):
    ax.plot(national[date_col], national[col], color=color, linewidth=2)
    ax.fill_between(national[date_col], national[col], alpha=0.15, color=color)
    ax.set_title(col.replace('_', ' ').title())
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('COVID-19 Indonesia — National Trends', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 3. Provincial Analysis

In [ ]:
# Find province column
prov_col = [c for c in df.columns if any(k in c.lower() for k in ['province', 'provinsi', 'location'])]
if prov_col:
    prov_col = prov_col[0]
    prov_summary = df.groupby(prov_col)[numeric_cols[0]].max().sort_values(ascending=False).head(15)
    
    prov_summary.plot(kind='barh', color='#c8a87a', figsize=(10, 7))
    plt.title(f'Top 15 Provinces by {numeric_cols[0].replace("_"," ").title()}')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print('No province column found — printing column names:', df.columns.tolist())

## 4. 7-Day Rolling Average (Smoothing)

In [ ]:
if len(numeric_cols) >= 1:
    col = numeric_cols[0]
    national_sorted = national.sort_values(date_col)
    national_sorted['rolling_7d'] = national_sorted[col].diff().rolling(7).mean()

    plt.figure(figsize=(14, 5))
    plt.bar(national_sorted[date_col], national_sorted[col].diff().clip(0),
            color='#e8d5b7', alpha=0.6, label='Daily new')
    plt.plot(national_sorted[date_col], national_sorted['rolling_7d'],
             color='#1a1a1a', linewidth=2, label='7-day rolling avg')
    plt.title(f'Daily New — {col.replace("_"," ").title()} with 7-Day Average')
    plt.legend()
    plt.tight_layout()
    plt.show()

## 5. Conclusions

- Indonesia experienced multiple waves of COVID-19, with peaks correlating with national events
- **Java (DKI Jakarta, East Java, West Java)** consistently reported the highest case counts
- The 7-day rolling average effectively smooths daily reporting inconsistencies for clearer trend analysis
- **Data quality note:** Daily reporting gaps and under-reporting in remote provinces should be considered when interpreting provincial comparisons